In [ ]:
import copy
import os.path as osp

import numpy as np
import pandas as pd
from calisim.data_model import (
	DistributionModel,
	ParameterDataType,
	ParameterSpecification,
)
from calisim.optimisation import OptimisationMethod, OptimisationMethodModel
from calisim.statistics import MeanSquaredError
from pcse.base import ParameterProvider
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from pcse.models import Wofost72_PP

In [ ]:
netherlands_wdp = NASAPowerWeatherDataProvider(latitude=51, longitude=5)
print(netherlands_wdp)

In [ ]:
india_wdp = NASAPowerWeatherDataProvider(latitude=23, longitude=73)
print(india_wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
netherlands_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_netherlands_2021.agro"))
print(netherlands_agro)

In [ ]:
india_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_india_2021.agro"))
print(india_agro)

In [ ]:
params = ParameterProvider(cropdata=cropd, sitedata=sited, soildata=soild)
wofost = Wofost72_PP(params, netherlands_wdp, netherlands_agro)
ground_truth = dict(
    TSUM1=255,
    TSUM2=1400,
    TBASEM=3.0,
    TSUMEM=170.0,
    TEFFMX=18.0,
    SPAN=37,
    TDWI=75,
    RGRLAI=0.016,
    Q10=2.0
)
for k in ground_truth:
    params.set_override(k, ground_truth[k])
wofost.run_till_terminate()
observed_data = pd.DataFrame(wofost.get_output())
observed_data

In [ ]:
parameter_spec = ParameterSpecification(
	parameters=[
		DistributionModel(
			name="TSUM1",
			distribution_name="uniform",
			distribution_args=[150, 280],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TSUM2",
			distribution_name="uniform",
			distribution_args=[1400, 2100],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TBASEM",
			distribution_name="uniform",
			distribution_args=[2, 4],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TSUMEM",
			distribution_name="uniform",
			distribution_args=[170, 255],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TEFFMX",
			distribution_name="uniform",
			distribution_args=[18, 32],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="SPAN",
			distribution_name="uniform",
			distribution_args=[20, 50],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TDWI",
			distribution_name="uniform",
			distribution_args=[75, 700],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="RGRLAI",
			distribution_name="uniform",
			distribution_args=[0.008, 0.02],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="Q10",
			distribution_name="uniform",
			distribution_args=[2, 3],
			data_type=ParameterDataType.CONTINUOUS,
		),
	]
)

In [ ]:
state_vars = ["LAI", "TWSO"]

def objective(
	parameters: dict, simulation_id: str, observed_data: np.ndarray | None,
    wdp: NASAPowerWeatherDataProvider, agro: list[dict]
) -> float | list[float]:
    p = copy.deepcopy(params)
    for k in parameters:
        p.set_override(k, parameters[k])

    wofost = Wofost72_PP(p, copy.deepcopy(wdp), copy.deepcopy(agro))
    wofost.run_till_terminate()
    simulated_data = pd.DataFrame(wofost.get_output())

    metric = MeanSquaredError()
    discrepancy = []
    for state_var in state_vars:
        simulated = simulated_data[state_var].values
        observed = observed_data[state_var].values

        if len(simulated) < len(observed):
            simulated = np.pad(simulated, (0, len(observed) - len(simulated)), mode='constant')

        discrepancy.append(metric.calculate(observed, simulated))

    return discrepancy

In [ ]:
import optuna
from optuna.importance import get_param_importances

optuna.logging.set_verbosity(optuna.logging.WARNING)

specification = OptimisationMethodModel(
	experiment_name="netherlands_optimisation",
	parameter_spec=parameter_spec,
	observed_data=observed_data,
	method="tpes",
	output_labels=state_vars,
	directions=["minimize", "minimize"],
	n_jobs=-1,
	n_iterations=250,
	calibration_func_kwargs=dict(wdp=netherlands_wdp, agro=netherlands_agro),
)

calibrator = OptimisationMethod(
	calibration_func=objective, specification=specification, engine="optuna"
)

calibrator.specify().execute().analyze()

In [ ]:
pd.DataFrame(dict(
    name=list(ground_truth.keys()),
    true=list(ground_truth.values())
)).merge(pd.DataFrame(calibrator.get_parameter_estimates().dict()["estimates"])).drop(columns="uncertainty")

In [ ]:
param_importances = {}

for i, state_var in enumerate(state_vars):
    importances = get_param_importances(calibrator.implementation.study, target=lambda t: t.values[i])
    param_importances[state_var] = dict(sorted(importances.items(), key=lambda x: x[1], reverse=True))

param_importances

In [ ]:
calibrator.implementation.study.trials_dataframe().rename(columns=dict(values_0="LAI", values_1="TWSO")).sort_values(state_vars)

In [ ]:
specification = OptimisationMethodModel(
	experiment_name="india_optimisation",
	parameter_spec=parameter_spec,
	observed_data=observed_data,
	method="tpes",
	output_labels=state_vars,
	directions=["minimize", "minimize"],
	n_jobs=-1,
	n_iterations=250,
	calibration_func_kwargs=dict(wdp=india_wdp, agro=india_agro),
)

calibrator = OptimisationMethod(
	calibration_func=objective, specification=specification, engine="optuna"
)

calibrator.specify().execute().analyze()

In [ ]:
pd.DataFrame(dict(
    name=list(ground_truth.keys()),
    true=list(ground_truth.values())
)).merge(pd.DataFrame(calibrator.get_parameter_estimates().dict()["estimates"])).drop(columns="uncertainty")

In [ ]:
param_importances = {}

for i, state_var in enumerate(state_vars):
    importances = get_param_importances(calibrator.implementation.study, target=lambda t: t.values[i])
    param_importances[state_var] = dict(sorted(importances.items(), key=lambda x: x[1], reverse=True))

param_importances

In [ ]:
calibrator.implementation.study.trials_dataframe().rename(columns=dict(values_0="LAI", values_1="TWSO")).sort_values(state_vars)